In [1]:
from dotenv import load_dotenv
import os
import requests
import json

In [2]:
load_dotenv()

True

In [3]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
model = "gemma-3-27b-it"

curl "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent"  
  -H 'Content-Type: application/json'  
  -H 'X-goog-api-key: GEMINI_API_KEY'  
  -X POST  
  -d '{
    "contents": [
      {
        "parts": [
          {
            "text": "Explain how AI works in a few words"
          }
        ]
      }
    ]
  }'

In [4]:
url = "https://generativelanguage.googleapis.com/v1beta/models/" + model + ":generateContent"
headers = {
    "Content-Type": "application/json",
    "X-goog-api-key": GEMINI_API_KEY
}
data = {
    "contents": [
        {
            "parts": [
                {
                    "text": "Explain how AI works in a few words"
                }
            ]
        }
    ]
}

In [5]:
response = requests.post(url, headers=headers, json=data)
response

<Response [200]>

In [6]:
print(response.json())

{'candidates': [{'content': {'parts': [{'text': 'AI learns from data to make decisions or predictions. \n\n(Essentially, it finds patterns and uses them!)\n\n\n\n'}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}], 'usageMetadata': {'promptTokenCount': 8, 'totalTokenCount': 8, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 8}]}, 'modelVersion': 'gemma-3-27b-it', 'responseId': 'JTevaP28EPiGz7IP2aPisQY'}


In [7]:
objetivo = "Aumentar la satisfacción del cliente en un 15% para el tercer trimestre de 2025, implementando un nuevo sistema de soporte en línea y capacitando al equipo de atención al cliente."

In [8]:
prompt = 'Actúa como evaluador de objetivos académicos de tesis de pregrado. Recibirás el texto de un objetivo y debes determinar si está correctamente formulado y redactado. Tu salida debe limitarse únicamente a un JSON válido, sin texto adicional fuera de él.\
Evalúa lo siguiente:\
1) El objetivo debe comenzar con un único verbo en infinitivo de la taxonomía de Bloom (ejemplo: “Diseñar”, “Implementar”, “Analizar”).;\
2) Puede haber varios verbos pero solo debe estar en infinitivo el principal o el del inicio. Otros verbos en la redacción son permitidos siempre que complementen el objetivo.;\
3) El objetivo debe responder de forma explícita a estas tres preguntas: ¿Qué? la acción principal del proyecto. ¿Cómo? el método, estrategia o acciones para lograrlo. ¿Para qué? el propósito o impacto esperado.;\
4) Toma en cuenta el contexto de la carrera (como diseño, producción u otras áreas afines). En estos casos, un objetivo puede no responder explícitamente a las tres preguntas, pero no necesariamente está mal. En tal situación, debes evaluar si el nivel de claridad es suficiente y aclararlo en el campo “detalle”.;\
5) Si alguno de los puntos falla, marca “aprobado” como “NO” y explica en el campo “detalle” qué falta o qué está incorrecto.;\
6) En el campo "sugerencias" detalla que puede mejorar del texto del objetivo, ya sea que fue aprobado o no.\
\
Cuando el objetivo no cumpla, debes también devolver 3 ejemplos de objetivos alternativos bien redactados (opciones de mejora).\
Si el objetivo cumple en todo, deja “opciones de sugerencias” como lista vacía.\
\
En el campo “verbos”, incluye cualquier verbo principal en infinitivo usado de manera incorrecta o mal escrito; si no hay problemas, devuelve una lista vacía. No inventes datos que no estén en el objetivo.\
\
El formato de salida debe ser exactamente este:\
\
{\
"aprobado": "SI" | "NO",\
"verbos": ["..."],\
"detalle": "",\
"sugerencias": "",\
"opciones de sugerencias": ["", "", ""]\
}\
\
Objetivo a evaluar: """' + objetivo + '"""\
\
Notas finales: No incluyas nada fuera del JSON. Si el objetivo es ambiguo o corresponde a un área en la que no siempre se expresan las tres preguntas, especifícalo en el campo “detalle” y sugiere en “sugerencias” cómo podría mejorar en claridad sin perder coherencia con la disciplina.'

In [9]:
prompt = '### ROL Y OBJETIVO ###\
Eres un evaluador experto de objetivos académicos para tesis de pregrado. Tu tarea es analizar un objetivo de tesis, determinar si está correctamente formulado según los criterios especificados y devolver tu evaluación en un formato JSON estricto.\
\
### CONTEXTO: TAXONOMÍA DE BLOOM ###\
El verbo principal del objetivo debe pertenecer a uno de los siguientes niveles cognitivos:\
- **CONOCIMIENTO/RECORDAR**: definir, listar, nombrar, identificar, memorizar.\
- **COMPRENSIÓN**: interpretar, resumir, clasificar, explicar, parafrasear.\
- **APLICACIÓN**: aplicar, usar, organizar, seleccionar, implementar.\
- **ANÁLISIS**: analizar, comparar, categorizar, deconstruir, distinguir.\
- **SÍNTESIS/CREAR**: crear, diseñar, planificar, construir, proponer, formular.\
- **EVALUACIÓN**: evaluar, juzgar, criticar, valorar, comprobar, justificar.\
\
### CRITERIOS DE EVALUACIÓN ###\
1.  **Verbo Inicial**: El objetivo DEBE comenzar con un único verbo en infinitivo de la taxonomía de Bloom.\
2.  **Verbos Secundarios**: Si existen otros verbos en la oración, no deben estar en infinitivo. Deben complementar la acción principal.\
3.  **Estructura S.M.A.R.T. Simplificada**: El objetivo debe responder claramente a tres preguntas:\
    * **¿Qué se hará?** (La acción principal, definida por el verbo).\
    * **¿Cómo se hará?** (El método, las herramientas o el proceso).\
    * **¿Para qué se hará?** (El propósito, la finalidad o el impacto esperado).\
4.  **Excepción de Contexto**: Si el objetivo no responde explícitamente a las 3 preguntas pero es inequívocamente claro dentro de un contexto disciplinar específico, puede ser aprobado. Debes anotar esta situación en el campo "detalle".\
\
### FORMATO DE SALIDA Y REGLAS ###\
Tu respuesta DEBE ser exclusivamente un objeto JSON válido y nada más. No incluyas texto introductorio, explicaciones adicionales ni la palabra `json` antes del código.\
\
- **`aprobado`**: "SI" o "NO". Marca "NO" si falla cualquiera de los criterios 1, 2 o 3 (y no aplica el 4).\
- **`verbos`**: Una lista de strings. Incluye cualquier verbo en infinitivo que esté mal utilizado (ej. un segundo infinitivo). Si no hay errores de verbos, deja la lista vacía `[]`.\
- **`detalle`**: Una explicación clara y concisa de por qué el objetivo fue aprobado o no. Si fue rechazado, especifica qué criterios fallaron.\
- **`sugerencias`**: Una recomendación general sobre cómo mejorar el objetivo, incluso si fue aprobado.\
- **`opciones de sugerencias`**: Si `aprobado` es "NO", proporciona 3 reescrituras completas y corregidas del objetivo. Si es "SI", deja la lista vacía `[]`.\
\
### EJEMPLOS (FEW-SHOT LEARNING) ###\
\
**Ejemplo 1: Objetivo BIEN formulado**\
"objetivo_a_evaluar": "Evaluar la efectividad de una campaña de marketing digital mediante el análisis de métricas de redes sociales para optimizar la inversión publicitaria."\
respuesta\
{\
    "aprobado": "SI",\
    "verbos": [],\
    "detalle": "El objetivo está correctamente formulado. Comienza con un verbo de evaluación (\'Evaluar\'), define el qué (\'la efectividad de una campaña\'), el cómo (\'mediante el análisis de métricas\') y el para qué (\'para optimizar la inversión\').",\
    "sugerencias": "Para mayor precisión, se podría especificar qué métricas clave se analizarán (ej. \'análisis de métricas como el CTR, tasa de conversión y alcance\').",\
    "opciones de sugerencias": []\
}\
\
**Ejemplo 2: Objetivo MAL formulado\
"objetivo_a_evaluar": "Se va a investigar y analizar los datos para poder comprender las tendencias del mercado."\
{\
    "aprobado": "NO",\
    "verbos": ["analizar", "comprender"],\
    "detalle": "El objetivo falla en varios criterios. No comienza con un verbo en infinitivo (inicia con \'Se va a investigar\'). Contiene múltiples verbos en infinitivo (\'analizar\', \'comprender\') que no deberían estarlo. No especifica el CÓMO (qué datos, con qué método) ni el PARA QUÉ de forma concreta.",\
    "sugerencias": "El objetivo debe reescribirse para que comience con un único verbo principal en infinitivo y especifique claramente el método y el propósito final de la investigación.",\
    "opciones de sugerencias": [\
    "Analizar los datos de ventas del último trimestre utilizando un modelo de regresión lineal para identificar las principales tendencias de consumo en el mercado local.",\
    "Interpretar los resultados de encuestas de satisfacción del cliente recopiladas durante 2024 para determinar los factores clave que influyen en la lealtad de marca.",\
    "Diagnosticar los patrones de comportamiento del consumidor en plataformas de e-commerce a través del análisis de datos de navegación para proponer mejoras en la experiencia de usuario."\
    ]\
}\
\
OBJETIVO A EVALUAR\
"""' + objetivo + '"""'

In [10]:
data = {
    "contents": [
        {
            "parts": [
                {
                    "text": prompt
                }
            ]
        }
    ]
}
response = requests.post(url, headers=headers, json=data)
response

<Response [200]>

In [11]:
response.json()["candidates"][0]["content"]["parts"][0]["text"][8:-4]

'{\n  "aprobado": "SI",\n  "verbos": [],\n  "detalle": "El objetivo está correctamente formulado. Comienza con un verbo de aplicación (\'Aumentar\'), define el qué (\'la satisfacción del cliente\'), el cómo (\'implementando un nuevo sistema de soporte en línea y capacitando al equipo\') y el para qué (\'para el tercer trimestre de 2025, en un 15%\').",\n  "sugerencias": "Para mayor claridad, se podría especificar las métricas que se utilizarán para medir la satisfacción del cliente (ej. \'encuestas de satisfacción\', \'net promoter score\').",\n  "opciones de sugerencias": []\n}'

In [ ]:
only_json = response.json()["candidates"][0]["content"]["parts"][0]["text"]
data = json.loads(only_json[only_json.find("{"):only_json.find("}") + 1])

In [13]:
data

{'aprobado': 'SI',
 'verbos': [],
 'detalle': "El objetivo está correctamente formulado. Comienza con un verbo de aplicación ('Aumentar'), define el qué ('la satisfacción del cliente'), el cómo ('implementando un nuevo sistema de soporte en línea y capacitando al equipo') y el para qué ('para el tercer trimestre de 2025, en un 15%').",
 'sugerencias': "Para mayor claridad, se podría especificar las métricas que se utilizarán para medir la satisfacción del cliente (ej. 'encuestas de satisfacción', 'net promoter score').",
 'opciones de sugerencias': []}